[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/user/repo/blob/main/notebook.ipynb)

# Lecture 15 & 16

 Decision making under certainty: crop choice (corn vs. spring barley)

A North Dakota farmer has 250 acres and a \$35,000$  loan for planting costs.

How many acres of corn ($X$) and barley ($Y$) maximize total
net profit?

**Objective function:**

$$
\max_{X,\,Y} \; (710 - 180)\,X + (400 - 100)\,Y
$$

**Constraints:**

$$
\begin{aligned}
X + Y &\le 250 && \text{(total acreage)} \\
180\,X + 100\,Y &\le 35{,}000 && \text{(initial investment)}
\end{aligned}
$$

**Lower and upper bounds:**

$$
0 \le X \le 250, \qquad 0 \le Y \le 250
$$



In [53]:
from scipy.optimize import linprog
import numpy as np

## 1. set up problem data

In [59]:
# ---------------------------------------------------------------------------
# Problem data
# ---------------------------------------------------------------------------
cost = np.array([180.0, 100.0])        # $/acre: corn, barley
revenue = np.array([710.0, 400.0])     # $/acre
net = revenue - cost                   # net profit $/acre -> [530, 300]
total_acres = 250.0
total_budget = 35_000.0

# Constaint matrix:  A @ [X, Y] <= b
A = np.array([[1.0, 1.0],              # X + Y <= 250
              cost])                   # 180X + 100Y <= 35,000
b = np.array([total_acres, total_budget])
bounds = [(0, total_acres), (0, total_acres)]



In [62]:
net

array([530., 300.])

In [60]:
## 2. Linear programming-- using linprog to minimize the NEGATIVE profit

In [57]:

res = linprog(c=-net, A_ub=A, b_ub=b, bounds=bounds, method="highs")
print("Linear programming (linprog):", res.x, -res.fun)

Linear programming (linprog): [125. 125.] 103750.0


In [66]:
np.sum(net*125)

np.float64(103750.0)

In [17]:
res

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -103750.0
              x: [ 1.250e+02  1.250e+02]
            nit: 2
          lower:  residual: [ 1.250e+02  1.250e+02]
                 marginals: [ 0.000e+00  0.000e+00]
          upper:  residual: [ 1.250e+02  1.250e+02]
                 marginals: [ 0.000e+00  0.000e+00]
          eqlin:  residual: []
                 marginals: []
        ineqlin:  residual: [ 0.000e+00  0.000e+00]
                 marginals: [-1.250e+01 -2.875e+00]
 mip_node_count: 0
 mip_dual_bound: 0.0
        mip_gap: 0.0

### -- hands on --

- a. Above Normal Condition

In [72]:
#revenue =      # $/acre
net = np.array([860.0, 320.0])
res_AN = linprog(c=-net, A_ub=A, b_ub=b, bounds=bounds, method="highs")
print("avoe Normal:", res_AN.x, -res_AN.fun)
res_AN

avoe Normal: [194.44444444   0.        ] 167222.22222222222


        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -167222.22222222222
              x: [ 1.944e+02  0.000e+00]
            nit: 3
          lower:  residual: [ 1.944e+02  0.000e+00]
                 marginals: [ 0.000e+00  1.578e+02]
          upper:  residual: [ 5.556e+01  2.500e+02]
                 marginals: [ 0.000e+00  0.000e+00]
          eqlin:  residual: []
                 marginals: []
        ineqlin:  residual: [ 5.556e+01  0.000e+00]
                 marginals: [-0.000e+00 -4.778e+00]
 mip_node_count: 0
 mip_dual_bound: 0.0
        mip_gap: 0.0

In [73]:
net[0]*194.4

np.float64(167184.0)

- b. Below Normal

In [75]:
#revenue =     # $/acre
net = np.array([290.0, 300.0])
res_BN = linprog(c=-net, A_ub=A, b_ub=b, bounds=bounds, method="highs")
print("Below Normal:", res_BN.x, -res_BN.fun)

Below Normal: [ -0. 250.] 75000.0


In [71]:
net

array([110., 200.])

## 2. pay off table

- single cell

In [23]:
### AbNormal Conditioin with Normal Plan
125*860+125*320

147500

- all cell at once

In [24]:
import pandas as pd

# Alternative actions: acres of each crop in each plan
plans = pd.DataFrame(
    {"Corn": [194, 125, 0], "Barley": [0, 125, 250]},
    index=["Plan AN", "Plan N", "Plan BN"],
)

# States of nature: net profit per acre ($)
net_profit = pd.DataFrame(
    {"Corn": [860, 530, 290], "Barley": [320, 300, 300]},
    index=["Above Normal", "Normal", "Below Normal"],
)

In [26]:
net_profit

,Corn,Barley
Above Normal,860,320
Normal,530,300
Below Normal,290,300


In [27]:
# Payoff table: rows = plans, columns = states of nature
payoff = plans @ net_profit.T
payoff

,Above Normal,Normal,Below Normal
Plan AN,166840,102820,56260
Plan N,147500,103750,73750
Plan BN,80000,75000,75000


## 3. regrate table


In [28]:
regret = payoff.max() - payoff
regret

,Above Normal,Normal,Below Normal
Plan AN,0,930,18740
Plan N,19340,0,1250
Plan BN,86840,28750,0


## 4. Expected Monetary Value (EMV)

$$
EMV(A_j) = \sum_{i=1}^{N} P[S_i]\, C(A_j, S_i)
$$

where $P[S_i]$ is the probability of state of nature $S_i$, and $C(A_j, S_i)$ is the monetary consequence of action $A_j$ under state $S_i$, with

$$
\sum_{i=1}^{N} P[S_i] = 1
$$

In [9]:
prob = pd.Series([1/3, 1/3, 1/3], index=payoff.columns)   # P[S_i], must sum to 1

emv = payoff @ prob
print("EMV ($)")
print(emv.round(0), "\n")
print("EMV decision:", emv.idxmax(), "\n")

EMV ($)
Plan AN    108640.0
Plan N     108333.0
Plan BN     76667.0
dtype: float64 

EMV decision: Plan AN 



## 4. Utility Function (linear)

The best outcome is assigned $U = 100$ and the worst outcome $U = 0$:

$$
U(x_{\max}) = 100, \qquad U(x_{\min}) = 0
$$

All other outcomes are scaled linearly in between:

$$
U(x) = 100 \times \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$

In [10]:
def utility_linear(x):
    return 100 * (x - x_min) / (x_max - x_min)

In [11]:
x_min = payoff.values.min()      # $56,260
x_max = payoff.values.max()      # $166,840

utility = utility_linear(payoff)
print("Utility table (linear)")
print(utility.round(1), "\n")

Utility table (linear)
         Above Normal  Normal  Below Normal
Plan AN         100.0    42.1           0.0
Plan N           82.5    42.9          15.8
Plan BN          21.5    16.9          16.9 



## 5. Expected Utility (EU)

$$
EU(A_j) = \sum_{i=1}^{N} P[S_i]\, U(A_j, S_i)
$$

where $U(A_j, S_i)$ is the utility of the outcome of action $A_j$ under state of nature $S_i$. The decision rule is to choose the action with the highest expected utility:

$$
A^{*} = \arg\max_{j} \; EU(A_j)
$$

In [12]:
eu = utility @ prob
print("Expected utility")
print(eu.round(1), "\n")
print("EU decision:", eu.idxmax())

Expected utility
Plan AN    47.4
Plan N     47.1
Plan BN    18.5
dtype: float64 

EU decision: Plan AN


### -- hands on --

**Risk avoider (square root, concave downward):**

$$
U(x) = 100 \times \sqrt{\frac{x - x_{\min}}{x_{\max} - x_{\min}}}
$$

# -- hands on --

## Crop decision model  

### 1. Starting from Normal rain = 452 for Cotton
- define variables

In [76]:

rainfall = 452
full_et = 700
cost = 9838
price = 35.5
ymax = 822



- Actual Yield

In [77]:
er  = 0.8 * rainfall - 10
fet = er / full_et
actual_yield = (0.8 * fet + 1.3 * fet**2 - 1.1 * fet**3) * ymax
print ('yield for Cotton under Normal rain',actual_yield, 'kg/ha')

yield for Cotton under Normal rain 485.31902897375164 kg/ha


- Net Profit

In [78]:
prof = price * actual_yield - cost
print('profit for Cotton under Normal Rain',prof,'Rs/ha')

profit for Cotton under Normal Rain 7390.825528568184 Rs/ha


### 2. expand the above to all four crops

In [79]:
# ---- Crop parameters: Cotton, Groundnuts, Pulses, Ragi ----
crops  = ["Cotton (Lc)", "Groundnuts (Lg)", "Pulses (Lp)", "Ragi (Lr)"]
full_et = np.array([700, 556, 400, 405])            # mm
cost    = np.array([9838, 11338, 5643, 4166])       # Rs/ha
price   = np.array([35.5, 13.81, 19.25, 4.39])      # Rs/kg
ymax    = np.array([800, 1800, 650, 1600])          # kg/ha

LAND       = 43934.0      # ha, total land
MAX_SHARE  = 0.70         # max fraction of land per crop
INVEST_HA  = 6000.0       # Rs/ha, initial-investment limit (averaged over total land)
USE_INVEST = True         # set False to drop the investment constraint

rain = {"Below Normal": 302, "Normal": 452, "Above Normal": 600}   # mm

In [80]:
def profit_per_ha(rainfall):
    """Net profit per hectare (Rs/ha) of each crop for a given rainfall (mm)."""
    er  = 0.8 * rainfall - 10
    fet = er / full_et
    actual_yield = (0.8 * fet + 1.3 * fet**2 - 1.1 * fet**3) * ymax
    return price * actual_yield - cost

- Only Normal Condition

In [81]:
rainfall = 452
c = profit_per_ha(rainfall)

## constrain A_ub @ (x,y,z,w) ≤ b_ub

In [82]:
A_ub = [[1,    1,     1,    1   ],    # total land
        [9838, 11338, 5643, 4166]]    # total investment
b_ub = [43934, 6000 * 43934]

- bounds

In [83]:
bounds = [(0, MAX_SHARE * LAND)] * 4
bounds

[(0, 30753.8), (0, 30753.8), (0, 30753.8), (0, 30753.8)]

In [84]:
res = linprog(c=-c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
res.x

array([ 9154.33081927,     0.        , 30753.8       ,     0.        ])

In [85]:
net_prof = np.sum(res.x * c)
net_prof


np.float64(259525917.89922345)

In [86]:
np.dot(res.x , c)

np.float64(259525917.89922345)

- expand to all rainfall

In [46]:
def optimal_plan(rainfall):
    # linprog minimizes, so negate profit to maximize it
    c = -profit_per_ha(rainfall)
    A_ub = [np.ones(4)]                 # sum(L) <= LAND
    A_ub.append(cost)      # sum(cost * L) <= 6000 * LAND
    b_ub = [LAND]
    b_ub.append(INVEST_HA * LAND)
    bounds = [(0, MAX_SHARE * LAND)] * 4
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
    if not res.success:
        raise RuntimeError(res.message)
    return res.x

In [47]:
def total_profit(plan, rainfall):
    return float(np.dot(plan, profit_per_ha(rainfall)))

In [48]:
#  ---- 1) Optimal plan for each rainfall category ----

In [49]:
rain.items()

dict_items([('Below Normal', 302), ('Normal', 452), ('Above Normal', 600)])

In [50]:
# -- optimal plan ---
for k, r in rain.items():
    print(k, crops, optimal_plan(r),np.int_(total_profit(optimal_plan(r),r)))

Below Normal ['Cotton (Lc)', 'Groundnuts (Lg)', 'Pulses (Lp)', 'Ragi (Lr)'] [ 6197.3895275     0.        30753.8        6982.8104725] 98001398
Normal ['Cotton (Lc)', 'Groundnuts (Lg)', 'Pulses (Lp)', 'Ragi (Lr)'] [ 9154.33081927     0.         30753.8            0.        ] 259525917
Above Normal ['Cotton (Lc)', 'Groundnuts (Lg)', 'Pulses (Lp)', 'Ragi (Lr)'] [26794.47042082     0.             0.             0.        ] 337742431
